In [1]:
import pandas as pd
import numpy as np
import requests
from pydantic import BaseModel, Field, field_validator


# ===== CONFIGURAÇÃO DO LOADER =====
class DataLoaderConfig(BaseModel):
    url: str = Field(..., description="URL do arquivo JSON")
    chave: str | None = Field(None, description="Chave do JSON a ser carregada")

    @field_validator("url")
    @classmethod
    def validar_url(cls, v: str) -> str:
        if not (v.startswith("http://") or v.startswith("https://")):
            raise ValueError("URL deve começar com http:// ou https://")
        return v


# ===== FUNÇÃO DE CARREGAMENTO =====
def load_data(config: DataLoaderConfig) -> pd.DataFrame:
    response = requests.get(config.url)
    response.raise_for_status()
    data = response.json()

    if isinstance(data, dict) and config.chave:
        data = data[config.chave]

    if isinstance(data, list):
        df = pd.json_normalize(data)
    elif isinstance(data, dict):
        df = pd.json_normalize([data])
    else:
        df = pd.DataFrame([data])

    return df


# ===== FUNÇÃO DE LIMPEZA INICIAL =====
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Corrige valores vazios em conta.cobranca.Total
    idx = df[df['conta.cobranca.Total'] == ' '].index
    df.loc[idx, "conta.cobranca.Total"] = (
        df.loc[idx, "conta.cobranca.mensal"] * 24
    )
    df.loc[idx, "cliente.tempo_servico"] = 24

    # Converte Total para float
    df['conta.cobranca.Total'] = df['conta.cobranca.Total'].astype(float)

    # Remove linhas sem valor em Churn
    df = df[df['Churn'] != ''].copy()
    df.reset_index(drop=True, inplace=True)

    return df


# ===== FUNÇÃO DE TRATAMENTO DE NULOS =====
def handle_nulls(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Remove duplicados
    df.drop_duplicates(inplace=True)

    # Preenche nulos em tempo_servico
    filtro = df['cliente.tempo_servico'].isna()
    df.loc[filtro, 'cliente.tempo_servico'] = np.ceil(
        df.loc[filtro, 'conta.cobranca.Total'] /
        df.loc[filtro, 'conta.cobranca.mensal']
    )

    # Remove nulos em colunas específicas
    colunas_dropar = [
        'conta.contrato',
        'conta.faturamente_eletronico',
        'conta.metodo_pagamento'
    ]
    df = df.dropna(subset=colunas_dropar).copy()
    df.reset_index(drop=True, inplace=True)

    return df


# ===== FUNÇÃO DE TRATAMENTO DE OUTLIERS =====
def treat_outliers(df: pd.DataFrame, column: str) -> pd.DataFrame:
    df = df.copy()

    # Passo 1: calcular limites iniciais
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # Passo 2: identificar outliers
    outliers_index = (df[column] < limite_inferior) | (df[column] > limite_superior)

    # Passo 3: tratar os outliers -> recalcular tempo_servico
    df.loc[outliers_index, column] = np.ceil(
        df.loc[outliers_index, 'conta.cobranca.Total'] /
        df.loc[outliers_index, 'conta.cobranca.mensal']
    )

    # Passo 4: recalcular limites
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    k= 1.5
    limite_inferior = Q1 - k * IQR
    limite_superior = Q3 + k * IQR

    # Passo 5: remover os que ainda forem outliers
    final_outliers_index = (df[column] < limite_inferior) | (df[column] > limite_superior)
    df = df[~final_outliers_index].copy()

    df.reset_index(drop=True, inplace=True)
    return df


# ===== PIPELINE COMPLETO =====
def process_pipeline(config: DataLoaderConfig) -> pd.DataFrame:
    df = load_data(config)
    df = clean_data(df)
    df = handle_nulls(df)
    df = treat_outliers(df, 'cliente.tempo_servico')
    return df


# ===== EXEMPLO DE USO =====
configs = {
    "churn": DataLoaderConfig(
        url="https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dataset-telecon.json",
        chave="dados_telecon"
    )
}

df_final = process_pipeline(configs["churn"])

print(f"Shape final do DataFrame: {df_final.shape}")
df_final.head()

Shape final do DataFrame: (7002, 21)


,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,0003-MKNFE,nao,masculino,0,nao,nao,9.0,sim,sim,DSL,...,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.9,542.40
1,0004-TLHLJ,sim,masculino,0,nao,nao,4.0,sim,nao,fibra otica,...,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.9,280.85
2,0011-IGKFF,sim,masculino,1,sim,nao,13.0,sim,nao,fibra otica,...,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.0,1237.85
3,0013-EXCHZ,sim,feminino,1,sim,nao,3.0,sim,nao,fibra otica,...,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.9,267.40
4,0013-MHZWF,nao,feminino,0,nao,sim,9.0,sim,nao,DSL,...,nao,nao,sim,sim,sim,mes a mes,sim,cartao de credito (automatico),69.4,571.45


In [2]:
df_sem_out = df_final.copy()

Mencionamos o problema de variáveis categóricas, mas antes de passarmos de fato para o notebook, vamos entender o que são essas variáveis.

##Trabalhando com variáveis categóricas

Imagine que você está realizando uma pesquisa sobre a cor dos olhos de algumas pessoas. Nesse contexto, a cor dos olhos é uma variável categórica, pois o valor que podemos atribuir a ela é se a cor é azul, verde, castanho…

Não conseguimos atribuir um valor numérico para essa variável, apenas classificá-la em categorias distintas.

Variáveis categóricas são uma forma de agrupar informações em categorias diferentes, sem um valor numérico associado a elas, como o caso das variáveis numéricas.

Compreendido o conceito, vamos acessar o notebook no Google Colab. Com o código df_sem_out digitado em uma célula, temos como retorno a tabela abaixo.

In [3]:
df_sem_out.head()

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,0003-MKNFE,nao,masculino,0,nao,nao,9.0,sim,sim,DSL,...,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.9,542.40
1,0004-TLHLJ,sim,masculino,0,nao,nao,4.0,sim,nao,fibra otica,...,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.9,280.85
2,0011-IGKFF,sim,masculino,1,sim,nao,13.0,sim,nao,fibra otica,...,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.0,1237.85
3,0013-EXCHZ,sim,feminino,1,sim,nao,3.0,sim,nao,fibra otica,...,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.9,267.40
4,0013-MHZWF,nao,feminino,0,nao,sim,9.0,sim,nao,DSL,...,nao,nao,sim,sim,sim,mes a mes,sim,cartao de credito (automatico),69.4,571.45


Perceba que temos uma primeira coluna chamada id_cliente. O ID é único para cada um dos 7.002 clientes registrados na nossa base de dados.

Essa é uma coluna categórica, mas não faz sentido mantê-la no nosso conjunto de dados, pois ela não nos traz nenhuma informação relevante para o modelo de machine learning.

Se inserirmos essa coluna no nosso modelo de machine learning, que trabalha com fórmulas matemáticas e cálculos estatísticos, ele irá procurar alguma relação na coluna.

Como ela é única, não há relação nenhuma, então haverá uma queda de desempenho, ou seja, demandará mais tempo para treinar o nosso modelo com uma informação que não é útil.

Vamos então remover a coluna id_cliente da base de dda base de dados.

#Substituindo valores

Na seção "Substituindo valores" abaixo do título "Trabalhando com variáveis categóricas", vamos digitar o dataframe df_sem_out seguido do método drop().

Esse método é utilizado para retirar algo. No caso, queremos retirar a coluna de ID dos clientes, então digitamos id_cliente entre aspas simples.

Como queremos retirar uma coluna, é necessário passar seu eixo. Digitamos axis após uma vírgula ainda dentro do método drop(), sendo igual a 1, ou seja, referente à coluna 1.



In [4]:
df_sem_out.drop('id_cliente', axis=1)

,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,internet.seguranca_online,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,nao,masculino,0,nao,nao,9.0,sim,sim,DSL,nao,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.90,542.40
1,sim,masculino,0,nao,nao,4.0,sim,nao,fibra otica,nao,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.90,280.85
2,sim,masculino,1,sim,nao,13.0,sim,nao,fibra otica,nao,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.00,1237.85
3,sim,feminino,1,sim,nao,3.0,sim,nao,fibra otica,nao,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.90,267.40
4,nao,feminino,0,nao,sim,9.0,sim,nao,DSL,nao,nao,nao,sim,sim,sim,mes a mes,sim,cartao de credito (automatico),69.40,571.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6997,nao,feminino,0,nao,nao,13.0,sim,nao,DSL,sim,nao,nao,sim,nao,nao,um ano,nao,cheque pelo correio,55.15,742.90
6998,sim,masculino,0,sim,nao,22.0,sim,sim,fibra otica,nao,nao,nao,nao,nao,sim,mes a mes,sim,cheque eletronico,85.10,1873.70
6999,nao,masculino,0,nao,nao,2.0,sim,nao,DSL,nao,sim,nao,nao,nao,nao,mes a mes,sim,cheque pelo correio,50.30,92.75
7000,nao,masculino,0,sim,sim,67.0,sim,nao,DSL,sim,nao,sim,sim,nao,sim,dois anos,nao,cheque pelo correio,67.85,4627.65


Testando o código dessa forma, teremos como resultado a mesma tabela, porém iniciada com a coluna churn em vez de id_cliente.

Era isso que queríamos, mas por enquanto apenas criamos um subconjunto do dataframe. Nosso objetivo é trabalhar com o novo dataframe, então no início do código anterior, vamos criar um novo dataframe chamado df_sem_id seguido de um sinal de igual (=).

Em seguida, incluímos o método copy() ao final do código. Dessa forma, o dataframe df_sem_id será independente do dataframe df_sem_out.

Para finalizar, digitamos novamente df_sem_id na segunda linha de código, apenas para obter a visualização de que o código funcionou corretamente. O resultado será uma tabela igual à anterior.

In [5]:
df_sem_id = df_sem_out.drop('id_cliente', axis=1).copy()
df_sem_id

,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,internet.seguranca_online,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,nao,masculino,0,nao,nao,9.0,sim,sim,DSL,nao,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.90,542.40
1,sim,masculino,0,nao,nao,4.0,sim,nao,fibra otica,nao,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.90,280.85
2,sim,masculino,1,sim,nao,13.0,sim,nao,fibra otica,nao,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.00,1237.85
3,sim,feminino,1,sim,nao,3.0,sim,nao,fibra otica,nao,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.90,267.40
4,nao,feminino,0,nao,sim,9.0,sim,nao,DSL,nao,nao,nao,sim,sim,sim,mes a mes,sim,cartao de credito (automatico),69.40,571.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6997,nao,feminino,0,nao,nao,13.0,sim,nao,DSL,sim,nao,nao,sim,nao,nao,um ano,nao,cheque pelo correio,55.15,742.90
6998,sim,masculino,0,sim,nao,22.0,sim,sim,fibra otica,nao,nao,nao,nao,nao,sim,mes a mes,sim,cheque eletronico,85.10,1873.70
6999,nao,masculino,0,nao,nao,2.0,sim,nao,DSL,nao,sim,nao,nao,nao,nao,mes a mes,sim,cheque pelo correio,50.30,92.75
7000,nao,masculino,0,sim,sim,67.0,sim,nao,DSL,sim,nao,sim,sim,nao,sim,dois anos,nao,cheque pelo correio,67.85,4627.65


Variáveis categóricas binárias
Agora estamos sem a coluna id_cliente, mas ainda há um problema: analisando nossas colunas restantes, percebemos que algumas possuem os valores nao ou sim, outras possuem masculino ou feminino, e outras os valores 0 e 1.

Essas colunas são chamadas variáveis categóricas binárias. Elas são um pouco diferentes das anteriores com três ou mais categorias distintas, pois estas possuem apenas duas.

Nosso objetivo é substituir os valores binários sim ou nao e masculino ou feminino, por 0 e 1. Modelos de machine learning trabalham muito melhor quando inserimos apenas números.

```
Alguns modelos trabalham com números exclusivamente.

Esse é o caso de modelos mais robustos.

```


Faremos o seguinte mapeamento:

*  De nao para 0;
*  De sim para 1;
*  De masculino para 0;
*  E de feminino para 1;


Para isso, vamos criar em uma nova célula a coluna mapeamento, que será igual a uma abertura de colchetes. Dentro dela, vamos digitar aspas simples uma linha abaixo e escrever nao, definindo-o com o valor 0.

Faremos o mesmo processo com as demais variáveis descritas acima na transcrição, chegando ao seguinte resultado:

In [6]:
mapeamento = {
    'nao': 0,
    'sim': 1,
    'masculino': 0,
    'feminino': 1
}

Temos o conjunto de mapeamento com os valores que serão utilizados para fazer a troca no conjunto de dados. Podemos executar a célula acima com o atalho "Shift + Enter".

Porém, existem variáveis categóricas com mais de duas categorias que possuem nao e sim. Nesse caso, podemos executar um código que já fizemos anteriormente na seção "Identificando e tratando strings vazias".

In [7]:
for col in df_sem_id.columns:
    print(f"Coluna: {col}")
    print(df_sem_id[col].unique())
    print("-" * 30)

Coluna: Churn
['nao' 'sim']
------------------------------
Coluna: cliente.genero
['masculino' 'feminino']
------------------------------
Coluna: cliente.idoso
[0 1]
------------------------------
Coluna: cliente.parceiro
['nao' 'sim']
------------------------------
Coluna: cliente.dependentes
['nao' 'sim']
------------------------------
Coluna: cliente.tempo_servico
[ 9.  4. 13.  3. 71. 63.  7. 66. 54. 72.  5. 56. 34.  1. 45. 50. 23. 55.
 26. 69. 37. 49. 67. 20. 43. 59. 12. 27.  2. 25. 29. 14. 35. 64. 39. 40.
 11.  6. 30. 70. 57. 58. 16. 32. 33. 10. 21. 61. 15. 44. 22. 24. 19. 47.
 62. 46. 52.  8. 60. 48. 28. 41. 53. 68. 31. 36. 17. 18. 65. 51. 38. 42.]
------------------------------
Coluna: telefone.servico_telefone
['sim' 'nao']
------------------------------
Coluna: telefone.varias_linhas
['sim' 'nao' 'sem servico de telefone']
------------------------------
Coluna: internet.servico_internet
['DSL' 'fibra otica' 'nao']
------------------------------
Coluna: internet.seguranca_onlin

Perceba que algumas colunas, como a cliente.parceiro, a cliente.dependentes, e a própria coluna Churn de saída, possuem os valores nao e sim.

Já colunas como cliente.genero, possuem os valores masculino e feminino, conforme mencionado anteriormente.

Porém, temos outras colunas como telefone.varias_linhas que possuem mais de duas opções, por exemplo, sim, nao e sem servico de telefone. Já a coluna internet.servico_internet tem os valores DSL, fibra otica e nao.

Nós não queremos alterar as colunas que possuem mais de duas categorias, pois elas são diferentes das colunas categóricas binárias. Teremos outro tratamento com esses tipos de colunas.

Para isso, vamos selecionar as colunas que possuem apenas nao e sim e fazer o mapeamento delas. As colunas são as listadas abaixo:

*   Churn
*   cliente.genero
*   cliente.parceiro
*   cliente.dependentes
*   telefone.servico_telefone
*   conta.faturamento_eletronico


Para selecioná-las, vamos utilizar o seguinte código:

In [8]:
colunas = ['telefone.servico_telefone', 'Churn', 'cliente.parceiro', 'cliente.dependentes', 'conta.faturamente_eletronico', 'cliente.genero']

Após executar a célula, teremos as colunas que queremos mapear. Agora vamos fazer de fato o mapeamento. Para isso, digitamos em uma nova célula o dataframe df_sem_id contendo entre colchetes a lista de colunas que acabamos de adicionar, ou seja, colunas.

Essa operação será igual a df_sem_id[colunas] seguida de um ponto (.) e do método replace(). Vamos passar para ele a variável mapeamento que criamos anteriormente.

Para finalizar, digitamos df_sem_id na linha de código abaixo.

In [9]:
df_sem_id[colunas] = df_sem_id[colunas].replace(mapeamento).infer_objects(copy=False)
df_sem_id

/tmp/ipython-input-3258575930.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sem_id[colunas] = df_sem_id[colunas].replace(mapeamento).infer_objects(copy=False)


,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,internet.seguranca_online,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,0,0,0,0,0,9.0,1,sim,DSL,nao,nao,nao,nao,nao,sim,mes a mes,0,cheque pelo correio,59.90,542.40
1,1,0,0,0,0,4.0,1,nao,fibra otica,nao,nao,sim,nao,nao,nao,mes a mes,1,cheque eletronico,73.90,280.85
2,1,0,1,1,0,13.0,1,nao,fibra otica,nao,sim,sim,nao,sim,sim,mes a mes,1,cheque eletronico,98.00,1237.85
3,1,1,1,1,0,3.0,1,nao,fibra otica,nao,nao,nao,sim,sim,nao,mes a mes,1,cheque pelo correio,83.90,267.40
4,0,1,0,0,1,9.0,1,nao,DSL,nao,nao,nao,sim,sim,sim,mes a mes,1,cartao de credito (automatico),69.40,571.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6997,0,1,0,0,0,13.0,1,nao,DSL,sim,nao,nao,sim,nao,nao,um ano,0,cheque pelo correio,55.15,742.90
6998,1,0,0,1,0,22.0,1,sim,fibra otica,nao,nao,nao,nao,nao,sim,mes a mes,1,cheque eletronico,85.10,1873.70
6999,0,0,0,0,0,2.0,1,nao,DSL,nao,sim,nao,nao,nao,nao,mes a mes,1,cheque pelo correio,50.30,92.75
7000,0,0,0,1,1,67.0,1,nao,DSL,sim,nao,sim,sim,nao,sim,dois anos,0,cheque pelo correio,67.85,4627.65


Perceba que nas primeiras colunas agora temos os valores 0 e 1 onde antes tínhamos nao e sim. O mesmo acontece nas colunas cliente.genero, cliente.parceiro, cliente.dependentes, e assim por diante.

Visualizando a parte direita da tabela, veremos que algumas colunas não mudaram e continuam com os valores anteriores.

Podemos conferir isso com mais propriedade a partir do código que escrevemos para imprimir os valores das colunas. Vamos copiá-lo e colar em uma nova célula logo após a transformação.

In [10]:
for col in df_sem_id.columns:
    print(f"Coluna: {col}")
    print(df_sem_id[col].unique())
    print("-" * 30)

Coluna: Churn
[0 1]
------------------------------
Coluna: cliente.genero
[0 1]
------------------------------
Coluna: cliente.idoso
[0 1]
------------------------------
Coluna: cliente.parceiro
[0 1]
------------------------------
Coluna: cliente.dependentes
[0 1]
------------------------------
Coluna: cliente.tempo_servico
[ 9.  4. 13.  3. 71. 63.  7. 66. 54. 72.  5. 56. 34.  1. 45. 50. 23. 55.
 26. 69. 37. 49. 67. 20. 43. 59. 12. 27.  2. 25. 29. 14. 35. 64. 39. 40.
 11.  6. 30. 70. 57. 58. 16. 32. 33. 10. 21. 61. 15. 44. 22. 24. 19. 47.
 62. 46. 52.  8. 60. 48. 28. 41. 53. 68. 31. 36. 17. 18. 65. 51. 38. 42.]
------------------------------
Coluna: telefone.servico_telefone
[1 0]
------------------------------
Coluna: telefone.varias_linhas
['sim' 'nao' 'sem servico de telefone']
------------------------------
Coluna: internet.servico_internet
['DSL' 'fibra otica' 'nao']
------------------------------
Coluna: internet.seguranca_online
['nao' 'sim' 'sem servico de internet']
---------

---

### Para saber mais: o que são variáveis categóricas?


Variáveis categóricas são usadas em Ciência de Dados para representar informações que podem ser divididas em grupos ou categorias. Elas não possuem uma escala numérica e servem para classificar dados de forma qualitativa.

Por exemplo, ao analisar o desempenho acadêmico de estudantes, podemos criar categorias como `"excelente"`, `"bom"` ou `"regular"`. Assim, é possível identificar padrões ou tendências no grupo analisado.

**Exemplos comuns de variáveis categóricas:**

* Cor dos olhos
* Tipo sanguíneo
* Marca de um carro
* Escolaridade

**Tipos de variáveis categóricas:**

1. **Nominais**
   Não possuem ordem ou hierarquia entre as categorias. Exemplo: preferências musicais — `"rock"`, `"jazz"`, `"pop"`.

2. **Ordinais**
   Possuem uma ordem específica entre as categorias. Exemplo: escolaridade — `"ensino fundamental completo"`, `"ensino médio completo"`, `"ensino superior completo"`.

3. **Binárias**
   Têm apenas duas categorias possíveis, como `"sim/não"`, `"verdadeiro/falso"` ou `"presente/ausente"`. São úteis para análises simples de distribuição entre duas opções.

---


---

### Para saber mais: IMPORTANTE - Mudança na versão do Pandas

A partir da versão **2.0.0 do Pandas** (lançada em 03/04/2023), houve uma mudança no valor **default** do parâmetro `dtype` do método `get_dummies()`. Antes, o padrão era `np.uint8`; agora, é `bool`.

Isso significa que ao criar variáveis dummies, você obterá `True` e `False` em vez de `1` e `0`, como no exemplo abaixo:

```python
import pandas as pd

s = pd.Series(list('abca'))
pd.get_dummies(s)
```

Saída:

| a     | b     | c     |
| ----- | ----- | ----- |
| True  | False | False |
| False | True  | False |
| False | False | True  |
| True  | False | False |

💡 **Observação:** Para os propósitos deste curso, isso **não afeta a preparação de dados para modelos de machine learning**, pois `True` equivale a `1` e `False` a `0`. Você pode deixar as variáveis dummies como `True`/`False` sem problemas.

Se quiser o mesmo resultado que era mostrado nas versões anteriores (0 e 1), basta definir explicitamente o `dtype=int`:

```python
pd.get_dummies(s, dtype=int)
```

Saída:

| a | b | c |
| - | - | - |
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |
| 1 | 0 | 0 |

Para mais detalhes, consulte a [documentação oficial do método `get_dummies()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html).

---


#Variáveis categóricas não binárias

##One Hot Encoder (ou Dummy)
A técnica que vamos aplicar pode ser chamada tanto de One Hot Encoder quanto de Dummy. Para evidenciar melhor como ela funciona, vamos utilizar um exemplo prático no notebook, dividido em duas células.

Na primeira célula da seção "One Hot Encoder (dummy)", temos o seguinte código:

In [11]:
s = pd.Series(list('abca'))
s

,0
0,a
1,b
2,c
3,a


```

O s colocado na segunda linha de código serve para visualizar o conteúdo dessa variável.
```

o executar a célula com "Shift + Enter", são retornadas quatro amostras com os índices de 0 a 3 e os respectivos valores de a, b, c e a.


Podemos pensar na variável de séries (s) como o dataframe de uma coluna, sendo os valores a, b, c e a as nossas categorias. Temos então 3 categorias distintas: a, b e c.

Agora vamos utilizar o método get_dummies() da biblioteca do Pandas. Para isso, digitamos o seguinte código em uma nova célula:


In [12]:
pd.get_dummies(s)

,a,b,c
0,True,False,False
1,False,True,False
2,False,False,True
3,True,False,False


O método get_dummies() é justamente a aplicação da técnica One Hot Encoder. Com ele, foi retornado um dataframe com quatro amostras, com os índices de 0 a 3 e três colunas: a, b e c.

A técnica utiliza as variáveis categóricas para criar novas colunas com os nomes das categorias distintas. Como existiam 3 categorias, foram criadas 3 colunas na tabela.

No índice 0, o valor correspondente à primeira coluna a é uma resposta para a seguinte pergunta: o índice 0 é a? Como o valor é 1, a resposta é sim, pois a era o valor presente no índice 0 da célula anterior.

Como as outras duas colunas não correspondem ao valor do índice 0, elas são preenchidas com o valor 0, que significa não.

Na variável s, o valor presente no índice 1 é a categoria b, então o valor 1 correspondente a “sim” é marcado na coluna b. Da mesma forma que antes, as demais colunas ficam com o valor 0, representando o “não”.

O processo se repete para as demais amostras de índice. É isso que o método get_dummies() faz: cria novas colunas e aplica 1 onde é indicado o valor do índice correspondente no dataframe original.

Iremos aplicar esse método no nosso conjunto de dados, mas há um detalhe muito importante: se você está utilizando a versão 1.0 do Pandas, assim como eu, serão obtidas as saídas 0 e 1.

Caso esteja utilizando a versão 2.0 ou mais do Pandas, a série obtida será com as saídas True e False.

Isso não tem nenhuma implicação prática no processo. Nosso objetivo é trabalhar com o conjunto de dados para inseri-los em um modelo de machine learning.

Quando as saídas são True e False, os modelos de machine learning interpretarão o True como 1 e o False como 0.

Se quiser obter o mesmo resultado utilizando uma versão diferente da 1.0, você pode adaptar a função pd.get_dummies() adicionando uma vírgula após a variável s e definindo o tipo (dtype) como int.

In [13]:
pd.get_dummies(s, dtype=int)

,a,b,c
0,1,0,0
1,0,1,0
2,0,0,1
3,1,0,0


Mas como o método get_dummies() faz o processamento? Ou seja, como ele sabe quais variáveis devem ser transformadas?

Em uma nova célula, vamos utilizar o dataframe anterior df_sem_id seguido do método info(). Executando a célula com "Shift + Enter", teremos o retorno abaixo:

In [14]:
df_sem_id.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7002 entries, 0 to 7001
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Churn                          7002 non-null   int64  
 1   cliente.genero                 7002 non-null   int64  
 2   cliente.idoso                  7002 non-null   int64  
 3   cliente.parceiro               7002 non-null   int64  
 4   cliente.dependentes            7002 non-null   int64  
 5   cliente.tempo_servico          7002 non-null   float64
 6   telefone.servico_telefone      7002 non-null   int64  
 7   telefone.varias_linhas         7002 non-null   object 
 8   internet.servico_internet      7002 non-null   object 
 9   internet.seguranca_online      7002 non-null   object 
 10  internet.backup_online         7002 non-null   object 
 11  internet.protecao_dispositivo  7002 non-null   object 
 12  internet.suporte_tecnico       7002 non-null   o

Perceba que existem alguns tipos de dtype, como:

*   int64
*   float64
*   object
As variáveis com dtype definido como object serão interpretadas como colunas categóricas. Vamos aplicar isso ao nosso conjunto de dados?

Em uma nova célula, digitaremos a mesma função pd.get_dummies() e passaremos para ela o dataframe df_sem_id, com o qual estamos trabalhando no momento.

In [20]:
pd.get_dummies(df_sem_id, dtype=int)

,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,conta.faturamente_eletronico,conta.cobranca.mensal,conta.cobranca.Total,...,internet.filmes_streaming_nao,internet.filmes_streaming_sem servico de internet,internet.filmes_streaming_sim,conta.contrato_dois anos,conta.contrato_mes a mes,conta.contrato_um ano,conta.metodo_pagamento_cartao de credito (automatico),conta.metodo_pagamento_cheque eletronico,conta.metodo_pagamento_cheque pelo correio,conta.metodo_pagamento_transferencia bancaria (automatica)
0,0,0,0,0,0,9.0,1,0,59.90,542.40,...,0,0,1,0,1,0,0,0,1,0
1,1,0,0,0,0,4.0,1,1,73.90,280.85,...,1,0,0,0,1,0,0,1,0,0
2,1,0,1,1,0,13.0,1,1,98.00,1237.85,...,0,0,1,0,1,0,0,1,0,0
3,1,1,1,1,0,3.0,1,1,83.90,267.40,...,1,0,0,0,1,0,0,0,1,0
4,0,1,0,0,1,9.0,1,1,69.40,571.45,...,0,0,1,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6997,0,1,0,0,0,13.0,1,0,55.15,742.90,...,1,0,0,0,0,1,0,0,1,0
6998,1,0,0,1,0,22.0,1,1,85.10,1873.70,...,0,0,1,0,1,0,0,1,0,0
6999,0,0,0,0,0,2.0,1,1,50.30,92.75,...,1,0,0,0,1,0,0,0,1,0
7000,0,0,0,1,1,67.0,1,0,67.85,4627.65,...,0,0,1,1,0,0,0,0,1,0


Note que foram criadas várias colunas. Originalmente, tínhamos 20 colunas. Agora temos 41 colunas, ou seja, foram criadas 21 colunas novas. Conforme esperado, a técnica cria uma coluna para cada categoria distinta.

Explorando a tabela, percebemos que os valores estão numéricos da forma como queríamos. Alguns estão com valores 0 ou 1, e outros com float.

Vamos observar agora as variáveis criadas. Em uma das colunas, temos a variável conta.contrato. Originalmente, essa era uma variável do nosso dataframe. Existia somente ela e dentro dela havia 3 valores: "dois anos", "mes a mes", e "um ano".

Com base nisso, foram criadas 3 novas colunas, uma para cada valor:

*   conta.contrato_dois anos
*   conta.contrato_mes a mes
*   conta.contrato_um ano

Vamos analisar a amostra do índice 0. A variável conta.contrato não correspondia ao valor de dois anos, então temos um 0 na coluna conta.contrato_dois anos.

Quanto à variável conta.contrato_mes a mes, o valor existente no índice 0 era de fato mês a mês, então temos a resposta 1 marcada nessa coluna.

Por fim, temos um 0 na coluna da variável conta.contrato_um ano, pois já sabemos que o valor no índice não é de um ano.

Isso se repete para todas as 21 variáveis dummies criadas.

##Salvando o dataframe

Há uma forma de visualizar todas as colunas criadas, mas antes vamos salvar o dataframe anterior. Como é esse o dataframe que vamos inserir no modelo de machine learning, é importante salvá-lo.

Iremos chamá-lo de df_dummies e ele será igual à função pd.get_dummies(df_sem_id). Lembrando que se você estiver utilizando a versão 2.0 ou superior do Pandas e quiser obter o resultado com os valores 0 e 1, é necessário incluir o dtype igual a int.

Vamos adicionar o método copy() ao final do código, para garantir que o dataframe com o qual estamos trabalhando seja totalmente independente.

Para finalizar, digitamos na segunda linha de código o nome do dataframe df_dummies seguido do método head() (separados por um ponto), para exibir somente as primeiras 5 linhas do conjunto de dados.

In [16]:
df_dummies = pd.get_dummies(df_sem_id, dtype=int).copy()
df_dummies.head()

,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,conta.faturamente_eletronico,conta.cobranca.mensal,conta.cobranca.Total,...,internet.filmes_streaming_nao,internet.filmes_streaming_sem servico de internet,internet.filmes_streaming_sim,conta.contrato_dois anos,conta.contrato_mes a mes,conta.contrato_um ano,conta.metodo_pagamento_cartao de credito (automatico),conta.metodo_pagamento_cheque eletronico,conta.metodo_pagamento_cheque pelo correio,conta.metodo_pagamento_transferencia bancaria (automatica)
0,0,0,0,0,0,9.0,1,0,59.9,542.40,...,0,0,1,0,1,0,0,0,1,0
1,1,0,0,0,0,4.0,1,1,73.9,280.85,...,1,0,0,0,1,0,0,1,0,0
2,1,0,1,1,0,13.0,1,1,98.0,1237.85,...,0,0,1,0,1,0,0,1,0,0
3,1,1,1,1,0,3.0,1,1,83.9,267.40,...,1,0,0,0,1,0,0,0,1,0
4,0,1,0,0,1,9.0,1,1,69.4,571.45,...,0,0,1,0,1,0,1,0,0,0


##Visualizando as colunas criadas

Para visualizar as colunas criadas, vamos digitar em uma nova célula o dataframe df_dummies seguido de columns separado por um ponto (.).

In [17]:
df_dummies.columns

Index(['Churn', 'cliente.genero', 'cliente.idoso', 'cliente.parceiro',
       'cliente.dependentes', 'cliente.tempo_servico',
       'telefone.servico_telefone', 'conta.faturamente_eletronico',
       'conta.cobranca.mensal', 'conta.cobranca.Total',
       'telefone.varias_linhas_nao',
       'telefone.varias_linhas_sem servico de telefone',
       'telefone.varias_linhas_sim', 'internet.servico_internet_DSL',
       'internet.servico_internet_fibra otica',
       'internet.servico_internet_nao', 'internet.seguranca_online_nao',
       'internet.seguranca_online_sem servico de internet',
       'internet.seguranca_online_sim', 'internet.backup_online_nao',
       'internet.backup_online_sem servico de internet',
       'internet.backup_online_sim', 'internet.protecao_dispositivo_nao',
       'internet.protecao_dispositivo_sem servico de internet',
       'internet.protecao_dispositivo_sim', 'internet.suporte_tecnico_nao',
       'internet.suporte_tecnico_sem servico de internet',
     

Perceba que agora temos uma lista bem maior.

Um exemplo de coluna criada foi telefone.varias_linhas_nao. telefone.varias_linhas era uma coluna original do nosso conjunto de dados, mas agora ela possui diversos valores, pois foram criadas novas colunas para cada categoria. São eles:

*   telefone.varias_linhas_nao
*   telefone.varias_linhas_sem servico de telefone
*   telefone.varias_linhas_sim

O processo se repetiu para as demais colunas, conforme indicado no retorno.

Para conferir se todas as variáveis estão numéricas, podemos digitar novamente o dataframe df_dummies seguido do método info().



In [18]:
df_dummies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7002 entries, 0 to 7001
Data columns (total 41 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Churn                                                       7002 non-null   int64  
 1   cliente.genero                                              7002 non-null   int64  
 2   cliente.idoso                                               7002 non-null   int64  
 3   cliente.parceiro                                            7002 non-null   int64  
 4   cliente.dependentes                                         7002 non-null   int64  
 5   cliente.tempo_servico                                       7002 non-null   float64
 6   telefone.servico_telefone                                   7002 non-null   int64  
 7   conta.faturamente_eletronico                                7002 non-null   int64  
 8 

No retorno, identificamos que todas as variáveis que possuímos agora são numéricas, com os tipos (dtype) int64 e float64.

Conclusão
Agora o nosso modelo de machine learning consegue trabalhar de forma mais otimizada e correta com os valores. Caso você leve o conjunto de dados para uma rede neural, ele só funcionará com variáveis numéricas, conforme queríamos.

Dessa forma, o modelo conseguirá captar as relações não lineares existentes entre as variáveis numéricas e as variáveis categóricas, para as quais foram criadas colunas dummies.

Nosso conjunto de dados está pronto para ser inserido em um modelo de machine learning!

---

### Para saber mais: parâmetros do `get_dummies()`

O método `get_dummies()` da biblioteca **Pandas** é utilizado para transformar variáveis categóricas em variáveis binárias.
Abaixo, os principais parâmetros:

* **`data`** (obrigatório): conjunto de dados com as variáveis categóricas.
* **`prefix`**: adiciona um prefixo às colunas geradas.

  * Ex.: `prefix="cat"` → colunas `cat_1`, `cat_2`, etc.
* **`prefix_sep`**: define o separador entre prefixo e nome da coluna.

  * Valor padrão: `"_"`.
* **`columns`**: seleciona quais colunas transformar (caso não seja definido, todas as categóricas serão convertidas).
* **`drop_first`**: remove a primeira coluna binária para evitar multicolinearidade.
* **`dtype`**: tipo de dado das colunas geradas (padrão = `uint8`, a partir da versão 2.0.0 passou a ser `bool`).

---

### Exemplo prático

```python
import pandas as pd

# Criando um DataFrame de exemplo
df = pd.DataFrame({
    'cor': ['vermelho', 'azul', 'verde', 'vermelho'],
    'tamanho': ['pequeno', 'médio', 'grande', 'médio'],
    'formato': ['quadrado', 'redondo', 'redondo', 'quadrado']
})

# Transformando colunas categóricas em variáveis dummies
df_dummies = pd.get_dummies(
    df,
    columns=['cor', 'tamanho'],
    prefix=['cor', 'tam'],
    prefix_sep='-',
    drop_first=True
)

print(df_dummies)
```

**Saída:**

| formato  | cor-verde | cor-vermelho | tam-médio | tam-pequeno |
| -------- | --------- | ------------ | --------- | ----------- |
| quadrado | 0         | 1            | 0         | 1           |
| redondo  | 0         | 0            | 1         | 0           |
| redondo  | 1         | 0            | 0         | 0           |
| quadrado | 0         | 1            | 1         | 0           |

---

✅ Nesse exemplo:

* `data` → o DataFrame `df`.
* `columns` → colunas que serão transformadas (`cor` e `tamanho`).
* `prefix` → adiciona prefixos (`cor-` e `tam-`).
* `prefix_sep='-'` → usa hífen como separador.
* `drop_first=True` → remove a primeira coluna binária de cada categoria para evitar redundância.

O resultado é um novo DataFrame (`df_dummies`) com as colunas originais + as variáveis binárias.

---

### 📋 Resumo dos principais parâmetros do `get_dummies()`

| Parâmetro        | Função                                                                       | Exemplo de uso                             |
| ---------------- | ---------------------------------------------------------------------------- | ------------------------------------------ |
| **`data`**       | Conjunto de dados com variáveis categóricas (obrigatório).                   | `pd.get_dummies(df)`                       |
| **`prefix`**     | Adiciona prefixo às colunas geradas.                                         | `prefix="cor"` → `cor_verde`, `cor_azul`   |
| **`prefix_sep`** | Define o separador entre prefixo e nome original (padrão = `_`).             | `prefix_sep="-"` → `cor-verde`, `cor-azul` |
| **`columns`**    | Define quais colunas transformar (caso não seja informado, aplica em todas). | `columns=["cor"]`                          |
| **`drop_first`** | Remove a primeira coluna de cada variável para evitar multicolinearidade.    | `drop_first=True`                          |
| **`dtype`**      | Tipo de dado das colunas geradas (`uint8` ou `bool`).                        | `dtype=int` → gera `0` e `1`               |

---

